# Part 3: Adversarial Attacks — Breaking the Classifier

**Objective:** Implement two adversarial attacks from scratch and measure how each degrades the classifier.

- **Attack 1:** Character-level evasion attack (perturb toxic comments to evade detection)
- **Attack 2:** Label-flipping poisoning attack (corrupt training labels to degrade the model)

In [ ]:
import os
import re
import random
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
THRESHOLD = 0.4
print("Libraries ready.")

## 3.1 Attack 1: Character-Level Evasion Attack

In [ ]:
# Homoglyph map: Latin → Cyrillic/other Unicode lookalikes
HOMOGLYPHS = {
    'a': '\u0430',  # Cyrillic а
    'e': '\u0435',  # Cyrillic е
    'o': '\u043e',  # Cyrillic о
    'p': '\u0440',  # Cyrillic р
    'c': '\u0441',  # Cyrillic с
    'x': '\u0445',  # Cyrillic х
    'y': '\u0443',  # Cyrillic у
    'i': '\u0456',  # Cyrillic і
}

ZERO_WIDTH_SPACE = '\u200B'

def insert_zero_width(word, every_n=2):
    """Insert zero-width space every `every_n` characters."""
    result = []
    for i, ch in enumerate(word):
        result.append(ch)
        if (i + 1) % every_n == 0 and i < len(word) - 1:
            result.append(ZERO_WIDTH_SPACE)
    return ''.join(result)

def apply_homoglyphs(word):
    """Replace Latin chars with Cyrillic lookalikes."""
    return ''.join(HOMOGLYPHS.get(ch.lower(), ch) for ch in word)

def duplicate_chars(word, prob=0.2):
    """Randomly duplicate 20% of characters."""
    result = []
    for ch in word:
        result.append(ch)
        if random.random() < prob:
            result.append(ch)  # duplicate
    return ''.join(result)

def perturb(text):
    """
    Apply three character-level perturbations to a text string:
    1. Zero-width space insertion
    2. Unicode homoglyph substitution
    3. Random character duplication (20%)
    """
    words = text.split()
    perturbed_words = []
    for word in words:
        w = insert_zero_width(word)
        w = apply_homoglyphs(w)
        w = duplicate_chars(w, prob=0.2)
        perturbed_words.append(w)
    return ' '.join(perturbed_words)

# Demonstrate
example = "You should kill yourself you worthless piece of garbage."
perturbed_example = perturb(example)
print("Original: ", example)
print("Perturbed:", perturbed_example)
print(f"\nOriginal length: {len(example)}, Perturbed length: {len(perturbed_example)}")

In [ ]:
# Load eval set and model
df_eval = pd.read_csv('eval_with_preds.csv')
print(f"Eval set: {df_eval.shape}")

tokenizer = AutoTokenizer.from_pretrained('./model_baseline')
model = AutoModelForSequenceClassification.from_pretrained('./model_baseline')
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model loaded on {device}")

In [ ]:
def get_proba_batch(texts, tokenizer, model, device, batch_size=64):
    """Get probability of class 1 (toxic) for a list of texts."""
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

# Select 500 comments: predicted toxic with confidence >= 0.7
df_eval['y_proba'] = df_eval['y_proba'].astype(float)
attack1_pool = df_eval[(df_eval['y_proba'] >= 0.7)].copy()
if len(attack1_pool) >= 500:
    attack1_sample = attack1_pool.sample(n=500, random_state=SEED)
else:
    attack1_sample = attack1_pool.sample(n=min(500, len(attack1_pool)), random_state=SEED)

print(f"Attack 1 sample size: {len(attack1_sample)}")
print(f"Average confidence (original): {attack1_sample['y_proba'].mean():.4f}")

In [ ]:
# Apply perturbation
print("Applying perturbations...")
perturbed_texts = [perturb(t) for t in attack1_sample['comment_text'].tolist()]

# Get predictions on perturbed texts
print("Running model on perturbed texts...")
perturbed_proba = get_proba_batch(perturbed_texts, tokenizer, model, device)
perturbed_pred  = (perturbed_proba >= THRESHOLD).astype(int)

# Original predictions (model predicted 1 for all, by construction)
original_proba = attack1_sample['y_proba'].values
original_pred  = np.ones(len(attack1_sample), dtype=int)

# Attack Success Rate: fraction that now get predicted 0
asr = (perturbed_pred == 0).mean()

print("\n" + "=" * 55)
print("ATTACK 1 RESULTS: Character-Level Evasion")
print("=" * 55)
print(f"Sample size:                    {len(attack1_sample)}")
print(f"Avg confidence (before):        {original_proba.mean():.4f}")
print(f"Avg confidence (after):         {perturbed_proba.mean():.4f}")
print(f"Attack Success Rate (ASR):      {asr:.4f} ({asr*100:.1f}%)")
print(f"  → {int(asr*len(attack1_sample))} of {len(attack1_sample)} toxic comments now evade detection")

In [ ]:
# Visualise confidence shift
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(original_proba, bins=30, color='steelblue', edgecolor='white', alpha=0.8, label='Before attack')
axes[0].hist(perturbed_proba, bins=30, color='crimson', edgecolor='white', alpha=0.6, label='After attack')
axes[0].axvline(THRESHOLD, color='black', linestyle='--', label=f'Threshold ({THRESHOLD})')
axes[0].set_xlabel('Model Confidence (P(toxic))')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Distribution: Before vs After Evasion Attack')
axes[0].legend()

attack1_table = pd.DataFrame({
    'Metric': ['Sample Size', 'Avg Confidence Before', 'Avg Confidence After',
               'Attack Success Rate', 'Comments Evading Detection'],
    'Value': [
        len(attack1_sample),
        f"{original_proba.mean():.4f}",
        f"{perturbed_proba.mean():.4f}",
        f"{asr:.4f} ({asr*100:.1f}%)",
        f"{int(asr*len(attack1_sample))} / {len(attack1_sample)}"
    ]
})
axes[1].axis('off')
tbl = axes[1].table(cellText=attack1_table.values, colLabels=attack1_table.columns,
                    loc='center', cellLoc='left')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 1.8)
axes[1].set_title('Attack 1 Summary Table', pad=20)

plt.tight_layout()
plt.savefig('part3_attack1.png', dpi=150)
plt.show()

## 3.2 Attack 2: Label-Flipping Poisoning Attack

In [ ]:
# Load the 100k training subset
df_train = pd.read_csv('train_subset.csv')
print(f"Training subset: {df_train.shape}")
print(f"Original label distribution:\n{df_train['label'].value_counts()}")

In [ ]:
# Poison: randomly flip 5% of labels
df_poisoned = df_train.copy()
n_flip = int(0.05 * len(df_poisoned))
flip_indices = np.random.choice(len(df_poisoned), size=n_flip, replace=False)
df_poisoned.loc[flip_indices, 'label'] = 1 - df_poisoned.loc[flip_indices, 'label']

n_changed = (df_poisoned['label'] != df_train['label']).sum()
print(f"Poisoned {n_changed} labels ({n_changed/len(df_train)*100:.1f}% of training set)")
print(f"Poisoned label distribution:\n{df_poisoned['label'].value_counts()}")

In [ ]:
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

MODEL_NAME = 'distilbert-base-uncased'  # Fresh checkpoint
tokenizer_p = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizing poisoned training data...")
poisoned_train_ds = ToxicDataset(df_poisoned['comment_text'], df_poisoned['label'], tokenizer_p)

df_eval_clean = pd.read_csv('eval_subset.csv')
print("Tokenizing evaluation data...")
eval_ds_p = ToxicDataset(df_eval_clean['comment_text'], df_eval_clean['label'], tokenizer_p)
print("Datasets ready.")

In [ ]:
# Train poisoned model
poisoned_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args_p = TrainingArguments(
    output_dir='./results_poisoned',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=200,
    evaluation_strategy='epoch',
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

def compute_metrics_p(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    from sklearn.metrics import roc_auc_score
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='macro'),
    }

trainer_p = Trainer(
    model=poisoned_model,
    args=training_args_p,
    train_dataset=poisoned_train_ds,
    eval_dataset=eval_ds_p,
    compute_metrics=compute_metrics_p,
)

print("Training poisoned model (from fresh distilbert-base-uncased checkpoint)...")
trainer_p.train()

In [ ]:
# Evaluate poisoned model on the clean evaluation set
pred_output_p = trainer_p.predict(eval_ds_p)
logits_p = pred_output_p.predictions
y_true_p = pred_output_p.label_ids
y_proba_p = torch.softmax(torch.tensor(logits_p), dim=-1)[:, 1].numpy()
y_pred_p  = (y_proba_p >= THRESHOLD).astype(int)

# Baseline metrics (from Part 1 — load or recompute)
baseline_proba = pd.read_csv('eval_with_preds.csv')['y_proba'].values
baseline_pred  = (baseline_proba >= THRESHOLD).astype(int)
y_true_base    = pd.read_csv('eval_with_preds.csv')['label'].values

def get_fnr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fn / (fn + tp)

results = pd.DataFrame([
    {
        'Model': 'Baseline (clean)',
        'Accuracy': round(accuracy_score(y_true_base, baseline_pred), 4),
        'F1 (macro)': round(f1_score(y_true_base, baseline_pred, average='macro'), 4),
        'FNR': round(get_fnr(y_true_base, baseline_pred), 4),
    },
    {
        'Model': 'Poisoned (5% label flip)',
        'Accuracy': round(accuracy_score(y_true_p, y_pred_p), 4),
        'F1 (macro)': round(f1_score(y_true_p, y_pred_p, average='macro'), 4),
        'FNR': round(get_fnr(y_true_p, y_pred_p), 4),
    }
])

print("=" * 60)
print("ATTACK 2 RESULTS: Label-Flipping Poisoning")
print("=" * 60)
print(results.to_string(index=False))
print(f"\nFNR increase: {results.iloc[1]['FNR'] - results.iloc[0]['FNR']:+.4f}")
print(f"F1 change:    {results.iloc[1]['F1 (macro)'] - results.iloc[0]['F1 (macro)']:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Before/after bar chart
metrics_to_plot = ['Accuracy', 'F1 (macro)', 'FNR']
x = np.arange(len(metrics_to_plot))
width = 0.35
baseline_vals = [results.iloc[0][m] for m in metrics_to_plot]
poisoned_vals = [results.iloc[1][m] for m in metrics_to_plot]

bars1 = axes[0].bar(x - width/2, baseline_vals, width, label='Baseline', color='#4CAF50', edgecolor='black')
bars2 = axes[0].bar(x + width/2, poisoned_vals, width, label='Poisoned', color='#F44336', edgecolor='black')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_to_plot)
axes[0].set_ylabel('Score'); axes[0].set_title('Attack 2: Before vs After Poisoning')
axes[0].legend(); axes[0].set_ylim(0, 1)
for bar in bars1: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)

# Summary table
axes[1].axis('off')
tbl = axes[1].table(cellText=results.values, colLabels=results.columns, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 2.0)
axes[1].set_title('Attack 2 Summary', pad=20)

plt.tight_layout()
plt.savefig('part3_attack2.png', dpi=150)
plt.show()

## 3.3 Operational Risk Analysis

### Which attack is more operationally dangerous for a live platform?

**Attack 1 (Character-level evasion)** is the higher **near-term operational risk**. In this experiment, ASR is 99.8% (499/500), indicating that toxic content can bypass automated detection with simple text transformations. This attack is highly accessible because adversaries can execute it directly at posting time without privileged system access.

**Attack 2 (Label-flipping poisoning)** has lower short-term exploitability because it requires access to the data or annotation pipeline. In this run, the measured degradation is real but modest (FNR +0.0075, macro-F1 -0.0144 at 5% label flips). However, poisoning remains a **systemic governance risk** because degradation can accumulate across retraining cycles and affect all future inferences after deployment.

### Which threat model is more realistic?

For a social media platform, **Attack 1 is generally more realistic** because:
1. Attackers can interact directly with the model via normal posting behavior.
2. No insider privileges are required.
3. Obfuscation scripts are widely available and easy to automate.

Attack 2 is still plausible, but typically requires insider abuse or compromise of data collection, moderation, or labeling workflows.

### Implication for Defense Prioritization

Defenses should be prioritized in this order:
1. **Input canonicalization at inference time**: strip zero-width characters, normalize Unicode, and apply confusable-character mapping or script-mixing checks before tokenization.
2. **Adversarial robustness hardening**: include obfuscated toxic variants in training/evaluation sets so the model learns invariance to these perturbations.
3. **Data pipeline integrity controls**: enforce provenance, reviewer quality checks, and drift/anomaly monitoring on label distributions to detect poisoning attempts early.
4. **Model governance**: keep canary sets, monitor safety metrics by cohort, and support fast rollback if post-deployment degradation is detected.

### Confidence and Limitations

These conclusions are based on one run and a finite sample. The risk ranking is operationally useful, but effect sizes should be validated with repeated seeds, larger attack sets, and uncertainty intervals.